### Time Series Trends & Anomalies

- The line chart above shows the monthly average temperature, with annotations for the warmest and coolest months.
- The bar chart displays monthly total precipitation, highlighting the peak rainy season.
- Comment on visible trends or anomalies in the markdown cell below after running the plots.

In [ ]:
# Monthly total precipitation
monthly_precip = df_clean.groupby([df_clean['date'].dt.to_period('M')])['PRECTOTCORR'].sum()
monthly_precip.index = monthly_precip.index.to_timestamp()
plt.figure(figsize=(12,5))
plt.bar(monthly_precip.index, monthly_precip.values, width=20)
plt.title('Monthly Total Precipitation (PRECTOTCORR) in Kenya')
plt.ylabel('Precipitation (mm)')
plt.xlabel('Month')
plt.grid(True)
# Annotate peak rainy months
peak_rain = monthly_precip.idxmax()
plt.annotate(f'Peak Rain: {peak_rain.strftime("%b %Y")}', xy=(peak_rain, monthly_precip.max()), xytext=(peak_rain, monthly_precip.max()+10), arrowprops=dict(arrowstyle='->'))
plt.show()

In [ ]:
# Monthly average T2M
monthly_t2m = df_clean.groupby([df_clean['date'].dt.to_period('M')])['T2M'].mean()
monthly_t2m.index = monthly_t2m.index.to_timestamp()
plt.figure(figsize=(12,5))
plt.plot(monthly_t2m, marker='o')
plt.title('Monthly Average Temperature (T2M) in Kenya')
plt.ylabel('Temperature (°C)')
plt.xlabel('Month')
plt.grid(True)
# Annotate warmest/coolest months
warmest = monthly_t2m.idxmax()
coolest = monthly_t2m.idxmin()
plt.annotate(f'Warmest: {warmest.strftime("%b %Y")}', xy=(warmest, monthly_t2m.max()), xytext=(warmest, monthly_t2m.max()+0.5), arrowprops=dict(arrowstyle='->'))
plt.annotate(f'Coolest: {coolest.strftime("%b %Y")}', xy=(coolest, monthly_t2m.min()), xytext=(coolest, monthly_t2m.min()-1), arrowprops=dict(arrowstyle='->'))
plt.show()

## 5. Time Series Analysis

We will plot monthly average temperature (T2M) and total precipitation (PRECTOTCORR), annotating the warmest/coolest and peak rainy months.

In [ ]:
# Export cleaned data
clean_path = 'data/kenya_clean.csv'
df_clean.to_csv(clean_path, index=False)
print(f"Cleaned data exported to {clean_path}")

In [ ]:
# Forward-fill weather variables
weather_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
df[weather_cols] = df[weather_cols].ffill()
# Drop rows with >30% missing values
row_missing_pct = df.isna().mean(axis=1)
df_clean = df.loc[row_missing_pct <= 0.3].copy()
print(f"Rows dropped due to >30% missing: {(row_missing_pct > 0.3).sum()}")
df_clean.reset_index(drop=True, inplace=True)

### Outlier Handling Decision

- Outliers are flagged above for each variable. For this analysis, we will retain outliers unless they are clearly data errors, as extreme values may represent real climate events.
- For missing values, we will forward-fill weather variables. If more than 30% of a row is missing, we will drop that row.

In [ ]:
# Outlier detection for key variables
z_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
z_scores = np.abs(stats.zscore(df[z_cols], nan_policy='omit'))
outlier_flags = (z_scores > 3)
outlier_counts = pd.Series(outlier_flags.sum(axis=0), index=z_cols)
outlier_counts

## 4. Outlier Detection & Basic Cleaning

We will compute Z-scores for key variables, flag outliers (|Z| > 3), and decide how to handle them. We will also handle remaining missing values.

### Interpretation of Summary Statistics and Missing Values

- The summary statistics above provide an overview of the distribution of each numeric variable.
- The missing value report lists columns with missing data and their percentage. Columns with >5% missing values should be noted for potential impact on analysis.
- Duplicate rows found and dropped: check the variable `duplicates` above for the count.

In [ ]:
# Missing value report
missing_report = df.isna().sum().to_frame('missing_count')
missing_report['missing_pct'] = 100 * missing_report['missing_count'] / len(df)
missing_report = missing_report[missing_report['missing_count'] > 0]
missing_report

In [ ]:
# Replace -999 with NaN
missing_before = df.isna().sum()
df.replace(-999, np.nan, inplace=True)
missing_after = df.isna().sum()

# Check for duplicates
duplicates = df.duplicated().sum()
df = df.drop_duplicates()

# Summary statistics
describe = df.describe()
describe

## 3. Summary Statistics & Missing-Value Report

We will replace all -999 values with NaN, check for duplicates, and generate summary statistics and missing value reports.

In [ ]:
# Load Kenya climate data
kenya_path = '../data/kenya.csv' if not os.path.exists('data/kenya.csv') else 'data/kenya.csv'
df = pd.read_csv(kenya_path)
df['Country'] = 'Kenya'
# Convert YEAR and DOY to datetime
df['date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['date'].dt.month
df.head()

## 2. Data Loading & Date Parsing

We will load the Kenya climate data, add a Country column, and convert YEAR and DOY to a proper datetime column. We will also extract the Month for seasonal analysis.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats
import os

# Kenya Climate Data EDA

This notebook performs exploratory data analysis (EDA), cleaning, and visualization for Kenya's climate dataset as part of the African Climate Trend Analysis challenge.

---

## 1. Import Required Libraries
We begin by importing the necessary Python libraries for data manipulation and visualization.

# Kenya Climate EDA
Task 2 notebook for Kenya (profiling, cleaning, time-series analysis, correlation, and distribution analysis).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

In [ ]:
country = 'Kenya'
df = pd.read_csv('../data/kenya.csv')
df['Country'] = country
df['DATE'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['DATE'].dt.month
df = df.replace(-999, np.nan)
dup_count = int(df.duplicated().sum())
df = df.drop_duplicates()
print('Duplicate rows found and removed:', dup_count)

In [ ]:
display(df.describe(include='number'))
missing = df.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / len(df)) * 100
display(missing.sort_values('missing_pct', ascending=False))

## Interpretation Notes
- Add a brief interpretation of key summary stats.
- List columns with >5% missing and explain analysis implications.

In [ ]:
z_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
z_input = df[z_cols].copy().fillna(df[z_cols].median(numeric_only=True))
z_abs = np.abs(zscore(z_input, nan_policy='omit'))
outlier_rows = (z_abs > 3).any(axis=1)
print('Outlier rows (|Z| > 3):', int(outlier_rows.sum()))

## Outlier Decision
Document whether you retain, cap, or drop outliers and why.

In [ ]:
row_missing_ratio = df.isna().mean(axis=1)
df = df[row_missing_ratio <= 0.30].copy()
weather_cols = ['T2M','T2M_MAX','T2M_MIN','T2M_RANGE','PRECTOTCORR','RH2M','WS2M','WS2M_MAX','PS','QV2M']
for col in weather_cols:
    if col in df.columns:
        df[col] = df[col].ffill()
df.to_csv('../data/kenya_clean.csv', index=False)
print('Saved cleaned file to ../data/kenya_clean.csv')

In [ ]:
monthly_temp = df.resample('M', on='DATE')['T2M'].mean()
warmest = monthly_temp.idxmax()
coolest = monthly_temp.idxmin()
plt.figure(figsize=(12, 4))
monthly_temp.plot()
plt.scatter([warmest, coolest], [monthly_temp.max(), monthly_temp.min()], color=['red', 'blue'])
plt.title('Monthly Average T2M - Kenya')
plt.ylabel('Temperature (°C)')
plt.show()

In [ ]:
monthly_prec = df.resample('M', on='DATE')['PRECTOTCORR'].sum()
peak_idx = monthly_prec.sort_values(ascending=False).head(3).index
plt.figure(figsize=(12, 4))
monthly_prec.plot(kind='bar')
for i in peak_idx:
    plt.axvline(i, color='orange', alpha=0.2)
plt.title('Monthly Total PRECTOTCORR - Kenya')
plt.ylabel('Precipitation (mm/month)')
plt.tight_layout()
plt.show()

## Time-Series Interpretation
Describe visible trends, anomalies, and likely peak rainy seasons.

In [ ]:
corr = df.select_dtypes(include='number').corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap - Kenya')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x='T2M', y='RH2M', ax=axes[0])
axes[0].set_title('T2M vs RH2M')
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', ax=axes[1])
axes[1].set_title('T2M_RANGE vs WS2M')
plt.tight_layout()
plt.show()

## Correlation Interpretation
Identify and explain the three strongest correlations.

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(df['PRECTOTCORR'], bins=50)
plt.yscale('log')
plt.title('PRECTOTCORR Distribution (log-y) - Kenya')
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='T2M', y='RH2M', size='PRECTOTCORR', sizes=(10, 200), alpha=0.5)
plt.title('Bubble: T2M vs RH2M (size = PRECTOTCORR) - Kenya')
plt.show()